In [5]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import chromadb

In [6]:
load_dotenv()

True

In [8]:
api_key = os.getenv("OPEN_API_KEY")
client = OpenAI(api_key=api_key)

print("OpenAI client created successfully!!")

OpenAI client created successfully!!


# Connect to API

In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the NVIDIA CEO Name?"},]
)
print(response.choices[0].message.content)

As of October 2023, the CEO of NVIDIA is Jensen Huang. He co-founded the company in 1993 and has been its CEO since then.


# Load data

In [12]:
with open("data/NVIDIA Corporation (NVDA) Presents at Bank of America 2026 Global Technology Conference Transcript.txt", "r") as f:
  bofa = f.read()

with open("data/NVIDIA Corporation (NVDA) Presents at Morgan Stanley Technology, Media & Telecom Conference 2026 Transcript.txt", "r") as f:
  morgan_stanley = f.read()

with open("data/NVIDIA Corporation (NVDA) Presents at NVIDIA GTC AI Conference 2026 Prepared Remarks Transcript.txt", "r") as f:
  gtc_ai = f.read()

with open("data/NVIDIA Corporation (NVDA) Presents at Second Annual AI Summit Transcript.txt", "r") as f:
  ai_summit = f.read()

with open("data/NVIDIA Corporation (NVDA) Presents at TD Cowen's 54th Annual Technology, Media & Telecom Conference Transcript.txt", "r") as f:
  td = f.read()
print(bofa)

NVIDIA Corporation (NVDA) Bank of America 2026 Global Technology Conference June 4, 2026 11:40 AM EDT

Company Participants

Colette Kress - Executive VP & CFO

Conference Call Participants

Vivek Arya - BofA Securities, Research Division

Presentation

Vivek Arya
BofA Securities, Research Division

Good morning. Welcome to day 3 of The Bank of America Global Technology Conference. I'm Vivek Arya, I cover semiconductor, semi cap equipment. And I'm really delighted and honored and it's a real treat to have Colette Kress, Executive Vice President and CFO of NVIDIA, to join us for the keynote session this morning, fresh off a number of announcements from GTC Taipei?

Question-and-Answer Session

Vivek Arya
BofA Securities, Research Division

So perhaps, Colette, if you could start with maybe giving us right, some sense of what NVIDIA announced, right? How it kind of fits your strategic direction, and then we can go into a few other questions. But thank you so much for joining us.

Colette

In [13]:
documents = {
    "BofA Conference": bofa,
    "Morgan Stanley Conference": morgan_stanley,
    "GTC AI Conference": gtc_ai,
    "AI Summit": ai_summit,
    "TD Cowen Conference": td,
}

for name, doc_text in documents.items():
    char_count = len(doc_text)
    word_count = len(doc_text.split())
    print(f"{name:28} | Chars: {char_count:,} | Words: {word_count:,}")

BofA Conference              | Chars: 30,782 | Words: 5,564
Morgan Stanley Conference    | Chars: 37,675 | Words: 6,633
GTC AI Conference            | Chars: 86,158 | Words: 15,002
AI Summit                    | Chars: 37,576 | Words: 6,706
TD Cowen Conference          | Chars: 26,746 | Words: 4,755


# Chunk Documents

In [14]:
import re

def clean_transcript_text(raw_text: str) -> str:
    """
    Cleans web copy-paste artifacts and normalizes line breaks.
    Ensures speaker names and major sections start on new paragraphs (\n\n).
    """
    # Normalize line breaks
    text = raw_text.replace("\r\n", "\n")

    pattern = r"(?m)^(Presentation|Question-and-Answer Session|Company Participants|Conference Call Participants|[A-Z][a-z]+(?:\s[A-Z][a-z]+)+)$"
    text = re.sub(pattern, r"\n\n\1", text)

    # 3. Collapse 3+ consecutive newlines down to standard double newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def chunk_document(text: str, source_name: str, max_chunk_size: int = 1200) -> list[dict]:
    """
    Splits transcript text into aggregated paragraph chunks.
    Filters out noise/boilerplate while keeping speaker names attached to responses.
    """
    cleaned_text = clean_transcript_text(text)
    paragraphs = cleaned_text.split("\n\n")

    # Boilerplate phrases/patterns to drop (adds noise to vector search)
    ignore_patterns = [
        "company participants",
        "conference call participants",
        "extel vote",
        "wi-fi password",
        "research division",
        "thank you, everybody",
        "welcome to day",
    ]

    chunks = []
    current_chunk = []
    current_length = 0

    for para in paragraphs:
        para = para.strip()

        # 1. Skip empty strings or explicit separators
        if not para or para.startswith("===="):
            continue

        # 2. Skip non-content introductory/conference administrative chatter
        lower_para = para.lower()
        if any(pattern in lower_para for pattern in ignore_patterns) and len(para) < 150:
            continue

        # 3. Aggregation Logic: If adding this block exceeds max size, save current chunk
        if current_length + len(para) > max_chunk_size and current_chunk:
            chunks.append({
                "text": "\n\n".join(current_chunk),
                "source": source_name
            })
            current_chunk = []
            current_length = 0

        current_chunk.append(para)
        current_length += len(para)

    # Append any remaining text
    if current_chunk:
        chunks.append({
            "text": "\n\n".join(current_chunk),
            "source": source_name
        })

    return chunks


# --- Execute Across All 5 Loaded Transcripts ---

documents = {
    "BofA Conference": bofa,
    "Morgan Stanley Conference": morgan_stanley,
    "GTC AI Conference": gtc_ai,
    "AI Summit": ai_summit,
    "TD Cowen Conference": td,
}

all_chunks = []

for source_name, text in documents.items():
    doc_chunks = chunk_document(text, source_name=source_name, max_chunk_size=1200)
    all_chunks.extend(doc_chunks)

print(f"Total Cleaned Chunks Created Across Corpus: {len(all_chunks)}")

Total Cleaned Chunks Created Across Corpus: 243


In [15]:
bofa_chunks = chunk_document(bofa, "BofA Conference")
morgan_stanley_chunks = chunk_document(morgan_stanley, "Morgan Stanley Conference")
gtc_ai_chunks = chunk_document(gtc_ai, "GTC AI Conference")
ai_summit_chunks = chunk_document(ai_summit, "AI Summit")
td_chunks = chunk_document(td, "TD Cowen Conference")

In [16]:
print(f"BofA Conference Chunks: {len(bofa_chunks)}")
print(f"Morgan Stanley Chunks: {len(morgan_stanley_chunks)}")
print(f"GTC AI Conference Chunks: {len(gtc_ai_chunks)}")
print(f"AI Summit Chunks: {len(ai_summit_chunks)}")
print(f"TD Cowen Chunks: {len(td_chunks)}")

print("-" * 35)
print(f"Total Combined Chunks: {len(all_chunks)}")

BofA Conference Chunks: 32
Morgan Stanley Chunks: 42
GTC AI Conference Chunks: 97
AI Summit Chunks: 42
TD Cowen Chunks: 30
-----------------------------------
Total Combined Chunks: 243


# Create Embeddings

In [17]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Extract the text string from each dictionary chunk in all_chunks
texts_to_encode = [chunk["text"] for chunk in all_chunks]

# Generate dense vector embeddings for all document chunks
embeddings = model.encode(texts_to_encode, show_progress_bar=True)

# Verify the embedding matrix shape and output
print("Embeddings Shape:", embeddings.shape)  # Expected: (num_chunks, 384)
print("Sample Embedding Vector:\n", embeddings[0])

C:\Users\bianc\PycharmProjects\EarningsCallRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 8/8 [00:07<00:00,  1.10it/s]

Embeddings Shape: (243, 384)
Sample Embedding Vector:
 [-4.58813645e-02  1.85638871e-02  3.78116481e-02 -7.98749458e-03
  2.21691970e-02  3.30308005e-02  3.03573087e-02  3.38680334e-02
 -1.78323761e-02 -2.06888281e-02 -1.03421926e-01 -1.80653147e-02
 -3.20616690e-03 -2.06364039e-02 -2.05488708e-02 -2.21075024e-02
  5.03515825e-02 -9.83099788e-02 -4.68211584e-02  3.69000435e-02
  1.26975225e-02 -4.60371524e-02  2.74941977e-02  5.14642743e-04
 -7.72009641e-02  8.25980771e-03 -3.79251479e-03  9.87257576e-04
 -2.66123880e-02 -4.78293933e-02 -6.80259317e-02  5.97695336e-02
  4.81230952e-02  2.24560853e-02  7.96680972e-02 -6.89491117e-03
 -1.65162352e-03 -3.60939279e-02 -4.98647392e-02 -5.00333048e-02
  3.37928906e-03 -1.04043134e-01 -3.20729762e-02  4.88762073e-02
  7.41342530e-02 -4.41028252e-02 -1.35732321e-02  1.62454974e-02
 -1.86106022e-02 -1.93602797e-02  4.97144787e-03 -1.19318686e-01
  1.35196466e-02 -6.53096884e-02 -7.38305831e-03  1.04394108e-01
 -3.94035205e-02 -1.49553835e-01  8

In [19]:
from sentence_transformers import util

# Select text directly from the "text" key of your chunk dictionary
chunk1_text = all_chunks[0]["text"]  # Intro / BofA Conference
chunk2_text = all_chunks[1]["text"]  # Keynote announcements (Vera Rubin / CPU)
chunk3_text = all_chunks[-1]["text"] # TD Cowen / Final chunk

# Generate embeddings for selected transcript chunks
embedding1 = model.encode(chunk1_text)
embedding2 = model.encode(chunk2_text)
embedding3 = model.encode(chunk3_text)

print("Embedding 1 Shape:", embedding1.shape)
print("Embedding 2 Shape:", embedding2.shape)
print("Embedding 3 Shape:", embedding3.shape)

# Check similarity scores between chunks
sim_1_2 = util.cos_sim(embedding1, embedding2)
sim_1_3 = util.cos_sim(embedding1, embedding3)

print("-" * 35)
print(f"Similarity (Chunk 1 vs Chunk 2): {sim_1_2.item():.4f}")
print(f"Similarity (Chunk 1 vs Chunk 3): {sim_1_3.item():.4f}")

Embedding 1 Shape: (384,)
Embedding 2 Shape: (384,)
Embedding 3 Shape: (384,)
-----------------------------------
Similarity (Chunk 1 vs Chunk 2): 0.4816
Similarity (Chunk 1 vs Chunk 3): 0.3562


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute cosine similarity matrices
similarity1 = cosine_similarity([embedding1], [embedding2])
similarity2 = cosine_similarity([embedding1], [embedding3])
similarity3 = cosine_similarity([embedding2], [embedding3])

# Print raw matrix outputs
print("Raw Similarity 1 vs 2:", similarity1)
print("Raw Similarity 1 vs 3:", similarity2)
print("Raw Similarity 2 vs 3:", similarity3)

print("-" * 35)

# Extract single floating-point values for cleaner reporting
print(f"Similarity (Chunk 1 vs Chunk 2): {similarity1[0][0]:.4f}")
print(f"Similarity (Chunk 1 vs Chunk 3): {similarity2[0][0]:.4f}")
print(f"Similarity (Chunk 2 vs Chunk 3): {similarity3[0][0]:.4f}")

Raw Similarity 1 vs 2: [[0.48155373]]
Raw Similarity 1 vs 3: [[0.35618317]]
Raw Similarity 2 vs 3: [[0.11217743]]
-----------------------------------
Similarity (Chunk 1 vs Chunk 2): 0.4816
Similarity (Chunk 1 vs Chunk 3): 0.3562
Similarity (Chunk 2 vs Chunk 3): 0.1122


# Storing Chunks with Chroma DB

In [25]:
# Initialize the ChromaDB client
chroma_client = chromadb.Client()

# Get or create the collection with the updated name
collection = chroma_client.get_or_create_collection(name="nvidia_transcripts")

print(f"Collection successfully created/loaded: {collection.name}")

Collection successfully created/loaded: nvidia_transcripts


In [26]:
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
    documents.append(chunk["text"])
    ids.append(f"chunk_{i}")
    metadatas.append({"source": chunk["source"]})

print(documents[0])
print(ids[0])
print(metadatas[0])

NVIDIA Corporation (NVDA) Bank of America 2026 Global Technology Conference June 4, 2026 11:40 AM EDT

Colette Kress - Executive VP & CFO

Presentation

Good morning. Welcome to day 3 of The Bank of America Global Technology Conference. I'm Vivek Arya, I cover semiconductor, semi cap equipment. And I'm really delighted and honored and it's a real treat to have Colette Kress, Executive Vice President and CFO of NVIDIA, to join us for the keynote session this morning, fresh off a number of announcements from GTC Taipei?

Question-and-Answer Session

So perhaps, Colette, if you could start with maybe giving us right, some sense of what NVIDIA announced, right? How it kind of fits your strategic direction, and then we can go into a few other questions. But thank you so much for joining us.

Colette Kress
Executive VP & CFO

Thank you for having me. I'm going to give you 1 quick statement at the very beginning here. As a reminder, this discussion may contain forward-looking statements, and 

In [27]:
collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

In [28]:
print(f" Stored {len(documents)} chunks in chroma db")
print(f" Sources: 5 different files")

 Stored 243 chunks in chroma db
 Sources: 5 different files


# Build Retrieval Pipeline

In [29]:
def retrieve(question, n_results = 3):
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )
  return results['documents'][0], results['metadatas'][0]

In [30]:
chunks, sources = retrieve("What drove NVIDIA's networking revenue growth and performance?", 3)

for i, (chunk, source) in enumerate(zip(chunks, sources)):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {source}")
    print(f"Text: {chunk}\n")

--- Chunk 1 ---
Source: {'source': 'GTC AI Conference'}
Text: This chart basically describes 100% of NVIDIA's strategies. You've been watching me talk about this slide from the very beginning. And ultimately, the single hardest thing to achieve is the thing on the bottom, installed base. It has taken us 20 years to now have built up hundreds of millions of GPUs and computing systems around the world that run CUDA. We are in every cloud, we're in every computer company. We serve just about every single industry. The installed base of CUDA is the reason why the flywheel is accelerating. The installed base is what attracts developers who then creates new algorithms that achieves a breakthrough. For example, deep learning. There are so many others. Those breakthroughs leads to entirely new markets, which builds new ecosystems around them with other companies that join, which creates a larger installed base.

--- Chunk 2 ---
Source: {'source': 'BofA Conference'}
Text: But if you think about

In [32]:
def ask_rag(question, n_results=3, verbose=True):
    chunks, sources = retrieve(question, n_results)

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"Question: {question}")
        print(f"{'─' * 60}")
        print(f"Retrieved {len(chunks)} chunks:")
        for i, (chunk, source) in enumerate(zip(chunks, sources)):
            # Handle source whether passed as dict or string
            source_name = source["source"] if isinstance(source, dict) else source
            print(f"   [{source_name}] {chunk[:80]}...")
        print(f"{'─' * 60}")

    context = "\n\n".join(chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions based ONLY on the provided context. "
                "If the context does not contain enough information to answer the question, "
                "say 'I don't have enough context or information to answer this question.' "
                "Do not make up information or assume anything. Strictly answer only from the provided context."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\n---\n\nQuestion: {question}",
        },
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,  # Low temperature for precise factual adherence
    )

    answer = response.choices[0].message.content

    if verbose:
        print(f"Answer: {answer}")
        print(f"{'═' * 60}")

    return answer

print("RAG Pipeline built successfully.")


RAG Pipeline built successfully.


In [33]:
ask_rag("What is NVIDIA's growth driver in networking and Spectrum-X Ethernet?")


════════════════════════════════════════════════════════════
Question: What is NVIDIA's growth driver in networking and Spectrum-X Ethernet?
────────────────────────────────────────────────────────────
Retrieved 3 chunks:
   [TD Cowen Conference] And that's the reason that we actually created Spectrum-X. And Spectrum-X is the...
   [GTC AI Conference] And as you will see later, because we continue to optimize the algorithms, and N...
   [TD Cowen Conference] Yes. Well, we can take an hour to answer this question. So if you have time, whe...
────────────────────────────────────────────────────────────
Answer: NVIDIA's growth driver in networking and Spectrum-X Ethernet is the need to provide a solution for AI workloads as data centers increasingly become AI factories. They recognize the importance of eliminating jitter and maintaining data order for efficient communication between GPUs, which is a challenge with traditional networking solutions. Additionally, NVIDIA's ability to optimi

"NVIDIA's growth driver in networking and Spectrum-X Ethernet is the need to provide a solution for AI workloads as data centers increasingly become AI factories. They recognize the importance of eliminating jitter and maintaining data order for efficient communication between GPUs, which is a challenge with traditional networking solutions. Additionally, NVIDIA's ability to optimize algorithms and leverage their large installed base allows them to reduce computing costs while increasing scale and speed. This is complemented by their integration with cloud services and OEMs, expanding their reach in the market."

# Create RAG Tool

In [34]:
def search_docs(query: str) -> str:
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    chunks = results['documents'][0]
    return "\n\n".join(chunks)

# JSON Schema for Tool Calling / Function Calling
rag_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search NVIDIA earnings call and investor conference transcripts (BofA, Morgan Stanley, GTC AI, AI Summit, TD Cowen) for relevant financial, product, hardware, and strategic insights.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to retrieve relevant transcript passages (e.g., networking revenue, Spectrum-X, Blackwell architecture, optical vs copper)."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

available_tools = {
    "search_docs": search_docs
}

print("RAG tool defined successfully.")

RAG tool defined successfully.


In [35]:
import json


def rag_agent(question, verbose=True):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to NVIDIA earnings call and investor conference transcripts "
                "(including BofA, Morgan Stanley, GTC AI, AI Summit, and TD Cowen). "
                "Use the search_docs tool to find answers from these transcripts. "
                "If the documents do not contain the answer, say so clearly. "
                "Always base your answers on the retrieved documents when using the tool. "
                "If tool use is not needed, specify that you are not using the tool and answer based on your knowledge."
            ),
        },
        {"role": "user", "content": question},
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"Question: {question}")
        print(f"{'─' * 60}")

    max_steps = 3
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=rag_tools,
        )

        choice = response.choices[0]

        # LLM completed generation
        if choice.finish_reason == "stop":
            if verbose:
                print(f"Answer: {choice.message.content}")
                print(f"{'═' * 60}")
            return choice.message.content

        # LLM requested tool call
        if choice.message.tool_calls:
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f'  Searching: "{args["query"]}"')

                result = available_tools[func_name](**args)

                if verbose:
                    print(f"  Found {len(result)} chars of context")

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    }
                )

    return "Could not answer within step limit."


print("RAG Agent ready.")

RAG Agent ready.


In [41]:
rag_agent("summarize what was talked about at the td cowen conference")


════════════════════════════════════════════════════════════
Question: summarize what was talked about at the td cowen conference
────────────────────────────────────────────────────────────
  Searching: "TD Cowen Conference"
  Found 2415 chars of context
Answer: At the TD Cowen Conference, Gilad Shainer from NVIDIA provided insights into the company's performance and the growth in their networking segment, which reported $14.9 billion, reflecting a 199% year-over-year increase. Shainer emphasized NVIDIA's role as a platform company, highlighting the significance of their technology and ecosystem in the broader landscape of artificial intelligence (AI). 

The conference featured a diverse range of participants and sessions covering various aspects of AI, including infrastructure, chips, platforms, models, and applications. Shainer also expressed gratitude to key technology figures who contributed to the event and recognized the participation of numerous companies, showcasing the exten

"At the TD Cowen Conference, Gilad Shainer from NVIDIA provided insights into the company's performance and the growth in their networking segment, which reported $14.9 billion, reflecting a 199% year-over-year increase. Shainer emphasized NVIDIA's role as a platform company, highlighting the significance of their technology and ecosystem in the broader landscape of artificial intelligence (AI). \n\nThe conference featured a diverse range of participants and sessions covering various aspects of AI, including infrastructure, chips, platforms, models, and applications. Shainer also expressed gratitude to key technology figures who contributed to the event and recognized the participation of numerous companies, showcasing the extensive reach and importance of AI technologies in modern industries."

In [47]:
rag_agent("What is the future of ai according to the conferences? Use the tool and site quotes from the conference transcripts")


════════════════════════════════════════════════════════════
Question: What is the future of ai according to the conferences? Use the tool and site quotes from the conference transcripts
────────────────────────────────────────────────────────────
  Searching: "future of AI"
  Found 2915 chars of context
  Searching: "AI trends 2024"
  Found 1930 chars of context
Answer: According to various insights from recent conferences, the future of AI is characterized by a transformative shift in computing paradigms and the integration of AI into various aspects of industries and everyday life.

1. **AI Understanding and Application**: One key aspect discussed is the necessity for AI to possess physical awareness and understanding of the world. For instance, CEOs like Jen-Hsun Huang emphasize that "AI needs to have physical awareness, physical understanding," which includes grasping concepts like causality and object permanence. This signifies that future AI systems will not only interpret data

'According to various insights from recent conferences, the future of AI is characterized by a transformative shift in computing paradigms and the integration of AI into various aspects of industries and everyday life.\n\n1. **AI Understanding and Application**: One key aspect discussed is the necessity for AI to possess physical awareness and understanding of the world. For instance, CEOs like Jen-Hsun Huang emphasize that "AI needs to have physical awareness, physical understanding," which includes grasping concepts like causality and object permanence. This signifies that future AI systems will not only interpret data but also understand real-world physical interactions and consequences.\n\n2. **AI as Part of the Corporate Framework**: An interesting shift in perspective highlighted by industry leaders is the integration of AI into corporate operations. Instead of maintaining a "human in the loop" approach, the idea is proposed that "every company should have AI in the loop." This i

In [49]:
rag_agent("what metrics are present in the transcripts? Give the exact numbers and give the context")


════════════════════════════════════════════════════════════
Question: what metrics are present in the transcripts? Give the exact numbers and give the context
────────────────────────────────────────────────────────────
  Searching: "metrics earnings"
  Found 3120 chars of context
  Searching: "financial metrics revenue earnings growth guidance"
  Found 2931 chars of context
Answer: The transcripts provide some insightful metrics and context regarding NVIDIA's financial performance and growth. Here are the key numbers and their context:

1. **Revenue Growth**: NVIDIA reported **$46 billion** in revenue for a recent quarter, marking a significant leap from previous figures and reflecting a dramatic growth trajectory. This revenue performance was highlighted as potentially the best earnings report in history, illustrating the company's remarkable achievement and the broad market impact of its innovations.

2. **Historical Context**: The CEO, Jensen Huang, remarked on how NVIDIA has tra

"The transcripts provide some insightful metrics and context regarding NVIDIA's financial performance and growth. Here are the key numbers and their context:\n\n1. **Revenue Growth**: NVIDIA reported **$46 billion** in revenue for a recent quarter, marking a significant leap from previous figures and reflecting a dramatic growth trajectory. This revenue performance was highlighted as potentially the best earnings report in history, illustrating the company's remarkable achievement and the broad market impact of its innovations.\n\n2. **Historical Context**: The CEO, Jensen Huang, remarked on how NVIDIA has transitioned over the years. The company's initial public offering (IPO) in 1998 was at **$48 million**, with trailing revenues of just **$30 million**. This comparison underscores the exponential growth NVIDIA has experienced, moving from millions in revenue in earlier years to billions today.\n\n3. **Market Potential and Strategic Insights**: Huang emphasized that current metrics s

In [51]:
ask_rag("what metrics were given at td cowen conference")


════════════════════════════════════════════════════════════
Question: what metrics were given at td cowen conference
────────────────────────────────────────────────────────────
Retrieved 3 chunks:
   [TD Cowen Conference] And of course, I want to try and optimize power consumption. I want to reduce po...
   [TD Cowen Conference] NVIDIA Corporation (NVDA) TD Cowen's 54th Annual Technology, Media & Telecom Con...
   [GTC AI Conference] But with this architecture, we're going to take our token generation speed, toke...
────────────────────────────────────────────────────────────
Answer: At the TD Cowen's 54th Annual Technology, Media & Telecom Conference, it was reported that NVIDIA's networking numbers were at $14.9 billion, which is up 199% year-over-year.
════════════════════════════════════════════════════════════


"At the TD Cowen's 54th Annual Technology, Media & Telecom Conference, it was reported that NVIDIA's networking numbers were at $14.9 billion, which is up 199% year-over-year."

In [52]:
ask_rag("summarize each conference")


════════════════════════════════════════════════════════════
Question: summarize each conference
────────────────────────────────────────────────────────────
Retrieved 3 chunks:
   [GTC AI Conference] But before I start, let me thank our pregame show hosts. I thought they did a gr...
   [BofA Conference] Maybe next year when you're here for the keynote, by that time.

Colette Kress
E...
   [BofA Conference] NVIDIA Corporation (NVDA) Bank of America 2026 Global Technology Conference June...
────────────────────────────────────────────────────────────
Answer: I don't have enough context or information to answer this question.
════════════════════════════════════════════════════════════


"I don't have enough context or information to answer this question."

In [53]:
ask_rag("what is trending in ai right now?")


════════════════════════════════════════════════════════════
Question: what is trending in ai right now?
────────────────────────────────────────────────────────────
Retrieved 3 chunks:
   [GTC AI Conference] Now our business already starting to show that. 60% of our business is hyperscal...
   [GTC AI Conference] An AI that was able to perceive became an AI that could generate. An AI that cou...
   [GTC AI Conference] But what happened in the last couple of years? Well, we've been watching, as you...
────────────────────────────────────────────────────────────
Answer: The current trend in AI is the shift towards deep learning and large language models, particularly in hyperscaler workloads. This includes advancements in generative AI, as exemplified by ChatGPT, which has initiated a new era of AI that can not only understand and perceive but also generate unique content. The demand for NVIDIA GPUs is also significantly increasing due to this trend.
═══════════════════════════════════

'The current trend in AI is the shift towards deep learning and large language models, particularly in hyperscaler workloads. This includes advancements in generative AI, as exemplified by ChatGPT, which has initiated a new era of AI that can not only understand and perceive but also generate unique content. The demand for NVIDIA GPUs is also significantly increasing due to this trend.'

In [54]:
rag_agent("What are hyperscaler workloads and what does nvidia mean when they talk about them")


════════════════════════════════════════════════════════════
Question: What are hyperscaler workloads and what does nvidia mean when they talk about them
────────────────────────────────────────────────────────────
  Searching: "hyperscaler workloads"
  Found 2954 chars of context
Answer: Hyperscaler workloads refer to the computing tasks managed by hyperscale data centers, which are typically operated by large cloud service providers (often referred to as "hyperscalers"). These workloads involve handling vast amounts of data and performing complex computations associated with cloud computing, artificial intelligence (AI), machine learning, and other high-demand applications.

In the context of NVIDIA, when they talk about hyperscaler workloads, they are highlighting their significant partnership with these large cloud providers to supply GPUs (Graphics Processing Units) and other technologies that are particularly well-suited for AI and deep learning tasks. They emphasize that a larg

'Hyperscaler workloads refer to the computing tasks managed by hyperscale data centers, which are typically operated by large cloud service providers (often referred to as "hyperscalers"). These workloads involve handling vast amounts of data and performing complex computations associated with cloud computing, artificial intelligence (AI), machine learning, and other high-demand applications.\n\nIn the context of NVIDIA, when they talk about hyperscaler workloads, they are highlighting their significant partnership with these large cloud providers to supply GPUs (Graphics Processing Units) and other technologies that are particularly well-suited for AI and deep learning tasks. They emphasize that a large portion of their business comes from these hyperscalers—roughly 60%, as mentioned in their communications. These workloads are increasingly shifting toward applications that leverage deep learning and large language models, signaling a growing demand for NVIDIA\'s GPUs, which excel in 